In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
len(ground_truth)

565

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
index = build_index(documents)

In [3]:
def text_search(query):
    boost_dict = {"question": 1.0, "answer": 2.0, "section": 0.1}

    return index.search(query, num_results=5, boost_dict=boost_dict)

In [4]:
from tqdm.auto import tqdm


def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])
    return [int(d["id"] == doc_id) for d in results]


def compute_relevance_total(ground_truth, search_function):
    return [compute_relevance(q, search_function) for q in tqdm(ground_truth)]

In [5]:
def hit_rate(relevance):
    return sum(1 for line in relevance if 1 in line) / len(relevance)


def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {"hit_rate": hit_rate(relevance_total), "mrr": mrr(relevance_total)}

In [6]:
evaluate(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.968141592920354, 'mrr': 0.8705899705014745}

In [7]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "answer": answer_boost,
        "section": section_boost,
    }
    return index.search(query, num_results=5, boost_dict=boost_dict)

In [8]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            result = evaluate(
                ground_truth,
                lambda query, qb=question_boost, ab=answer_boost, sb=section_boost: search_boosts(query, qb, ab, sb),
            )
            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

...

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

In [9]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
35,5.0,10.0,0.5,0.968142,0.870590
3,1.0,2.0,0.1,0.968142,0.870590
19,2.0,4.0,0.2,0.968142,0.870590
18,2.0,4.0,0.1,0.969912,0.869322
4,1.0,2.0,0.2,0.966372,0.868879
20,2.0,4.0,0.5,0.964602,0.868525
34,5.0,10.0,0.2,0.969912,0.868437
7,1.0,4.0,0.2,0.973451,0.865251
33,5.0,10.0,0.1,0.969912,0.864602
8,1.0,4.0,0.5,0.961062,0.860767
